# 06 — Transfer Learning (EfficientNetB0, ResNet50, InceptionV3)

**Objectif (exigence jury 2)** : implementer ET fine-tuner au moins 2 modeles pre-entraines sur ImageNet. Nous en entrainons **3** (3 familles distinctes) selon un protocole de fine-tuning en **2 phases**.

**A executer de preference sur Google Colab (GPU).** Adapter `PROJECT_DIR` dans la cellule d'amorcage si besoin (chemin Google Drive).

Pipeline de donnees, modeles et evaluation sont importes depuis `src/` pour garder le notebook lisible et reproductible.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# --- Portabilite Colab / local ---
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Montez votre Drive et placez-y le projet, puis ajustez ce chemin.
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/covid_19_radiographie"  # <-- a adapter
    os.environ["COVID_BASE_DIR"] = PROJECT_DIR
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
    # Installe les dependances manquantes sur Colab
    !pip -q install shap
else:
    # En local : racine = parent du dossier notebooks/
    PROJECT_DIR = os.path.abspath("..")
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)

from src.utils.env import setup_environment, ensure_dirs, get_paths
INFO = setup_environment()
PATHS = ensure_dirs()
print("Projet :", PROJECT_DIR)

## 0. Mise en place des donnees

Le dataset (21 165 images) doit etre accessible sous `<PROJECT_DIR>/data/COVID-19_Radiography_Dataset/<CLASSE>/images/`.

**Trois options sur Colab** :
1. Donnees deja sur votre Google Drive (monte par la cellule precedente) — rien a faire.
2. **Telechargement Kaggle** (recommande, rapide) : deposez votre `kaggle.json` (jeton API Kaggle) puis executez la cellule ci-dessous.
3. Upload manuel du dossier sur Drive (lent).

La cellule verifie la presence des donnees et, si absentes sur Colab, propose le telechargement Kaggle.

In [ ]:
from pathlib import Path
data_dir = PATHS['data']
ok = (data_dir / 'COVID' / 'images').is_dir()
print('Donnees presentes :', ok, '->', data_dir)

if not ok and IN_COLAB:
    print('Tentative de telechargement depuis Kaggle...')
    from google.colab import files
    import os, json
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print('Deposez votre kaggle.json :')
        up = files.upload()  # selectionnez kaggle.json
        os.makedirs('/root/.kaggle', exist_ok=True)
        for name in up: os.replace(name, '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip -q install kaggle
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d tawsifurrahman/covid19-radiography-database -p {str(data_dir.parent)}
    import zipfile, glob
    zips = glob.glob(str(data_dir.parent / '*.zip'))
    if zips:
        with zipfile.ZipFile(zips[0]) as z: z.extractall(data_dir.parent)
        print('Extraction terminee.')
    # Le zip Kaggle extrait un dossier 'COVID-19_Radiography_Dataset' : verifier le chemin
    ok = (data_dir / 'COVID' / 'images').is_dir()
    print('Donnees presentes apres telechargement :', ok)

assert (data_dir / 'COVID' / 'images').is_dir(), (
    f'Dataset introuvable sous {data_dir}. Placez-le sur Drive ou via Kaggle.')

## 1. Donnees : split stratifie fige 70/15/15

Le split est realise sur les **chemins de fichiers** (aucune fuite) et persiste dans `reports/splits/`. **Tous** les modeles du projet seront evalues sur ce meme test set -> comparaison defendable. Le sous-dossier `masks/` est exclu, les images niveaux de gris sont converties en RGB 3 canaux (poids ImageNet).

In [ ]:
from src.data import tf_pipeline as tp

splits = tp.build_split_csvs()
print({k: len(v) for k, v in splits.items()})
display(tp.split_summary())
class_weights = tp.get_class_weights()
print('Poids de classe (balanced, sur train) :', class_weights)

## 2. Protocole de fine-tuning en 2 phases

- **Phase 1 (extraction)** : backbone gele, on entraine seulement la tete `GAP -> Dropout -> Dense(256) -> Dropout -> softmax`, Adam `lr=1e-3`.
- **Phase 2 (fine-tuning)** : on degele le dernier bloc du backbone, on **garde les BatchNorm gelees** (sinon divergence), on **recompile** (obligatoire) avec Adam `lr=1e-5`.

Callbacks : `ModelCheckpoint`, `EarlyStopping(restore_best_weights)`, `ReduceLROnPlateau`, `CSVLogger` (historique persiste pour les courbes d'apprentissage du notebook 08).

On utilise les **poids de classe** pour gerer le desequilibre.

In [ ]:
import numpy as np
from src.models import transfer_learning as tl
from src.models import evaluation as ev

def run_backbone(backbone, batch_size=None, epochs_phase1=12, epochs_phase2=15):
    spec = tl.get_spec(backbone)
    bs = batch_size or INFO['default_batch_size']
    if spec.img_size == 299:
        bs = max(8, bs // 2)  # Inception @299 plus gourmand
    train_ds = tp.make_dataset('train', spec.img_size, spec.preprocess_fn, bs, augment=True)
    val_ds   = tp.make_dataset('val',   spec.img_size, spec.preprocess_fn, bs)
    test_ds  = tp.make_dataset('test',  spec.img_size, spec.preprocess_fn, bs, shuffle=False)

    model, h1, h2 = tl.train_tl_model(
        backbone, train_ds, val_ds, class_weights, PATHS,
        epochs_phase1=epochs_phase1, epochs_phase2=epochs_phase2)

    # Evaluation sur le test set fige
    y_true = tp.get_labels('test')
    y_proba = ev.predict_keras(model, test_ds)
    y_pred = y_proba.argmax(axis=1)
    metrics = ev.evaluate_model(backbone, y_true, y_pred, y_proba, save_dir=PATHS['metrics'])
    ev.print_metrics_summary(metrics)
    ev.plot_confusion(metrics, save_path=PATHS['figures'] / f'confusion_{backbone}.png')
    ev.plot_roc_pr_ovr(y_true, y_proba, save_path=PATHS['figures'] / f'roc_pr_{backbone}.png', title=backbone)
    return model, metrics

## 3. EfficientNetB0 (le plus efficace — valide le pipeline)

In [ ]:
eff_model, eff_metrics = run_backbone('efficientnetb0')

## 4. ResNet50

In [ ]:
res_model, res_metrics = run_backbone('resnet50')

## 5. InceptionV3 (entree native 299x299)

In [ ]:
inc_model, inc_metrics = run_backbone('inceptionv3')

## 6. Recapitulatif

Les modeles sont sauvegardes dans `models/transfer/<backbone>_best.keras`, les historiques dans `reports/history/`, les metriques dans `reports/metrics/`. La comparaison complete est faite dans le notebook **08-comparaison**.

In [ ]:
import pandas as pd
rows = [eff_metrics, res_metrics, inc_metrics]
pd.DataFrame([{ 'modele': m['model'], 'f1_macro': m['f1_macro'],
  'balanced_acc': m['balanced_accuracy'], 'rappel_COVID': m['per_class']['COVID']['recall']}
  for m in rows]).sort_values('f1_macro', ascending=False)